In [ ]:
# ===============================================================
# 🧪 Week 1 — Catalyst Dataset Exploration
# Project: HQNN for Catalyst Performance Prediction
# Author: Taofeek Sanyaolu
# ===============================================================

import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from fairchem.core.datasets.lmdb_dataset import LmdbDataset
from ase import Atoms
from ase.visualize.plot import plot_atoms

# === Configure paths ===
data_dir = r"E:\Computational Engineering\samples\is2res_lmdbs\is2res_train_val_test_lmdbs\data\is2re\100k\train"
config = {"src": data_dir, "train": True}
ds = LmdbDataset(config)

print(f"✅ Dataset loaded successfully — {len(ds)} samples")
print("Available keys:", list(ds[0].keys()))

# === Helper: extract numerical fields ===
def extract_numeric(key):
    vals = []
    for i in range(min(len(ds), 5000)):  # sample first 5k
        sample = ds[i]
        if key in sample:
            arr = np.array(sample[key])
            if arr.ndim == 0:
                vals.append(arr.item())
            else:
                vals.append(arr.mean())
    return np.array(vals)

# ---------------------------------------------------------------
# 1️⃣ Atomic Number Distribution
# ---------------------------------------------------------------
atoms = [sample["atomic_numbers"] for sample in ds[:500]]
flat_atoms = np.concatenate(atoms)
unique, counts = np.unique(flat_atoms, return_counts=True)
plt.figure(figsize=(10, 4))
sns.barplot(x=unique, y=counts, color="teal")
plt.title("Atomic Number Frequency (first 500 samples)")
plt.xlabel("Atomic Number (Z)")
plt.ylabel("Count")
plt.show()

# ---------------------------------------------------------------
# 2️⃣ Energy Distribution (y_init vs y_relaxed)
# ---------------------------------------------------------------
y_init = extract_numeric("y_init")
y_relaxed = extract_numeric("y_relaxed")

plt.figure(figsize=(10, 5))
sns.histplot(y_init, bins=50, color="orange", label="y_init", kde=True)
sns.histplot(y_relaxed, bins=50, color="blue", label="y_relaxed", kde=True)
plt.legend()
plt.title("Energy Distribution — Initial vs Relaxed")
plt.xlabel("Energy (eV)")
plt.ylabel("Frequency")
plt.show()

# ---------------------------------------------------------------
# 3️⃣ Force Magnitude Distribution
# ---------------------------------------------------------------
forces = [np.linalg.norm(sample["force"], axis=1).mean() for sample in ds[:1000]]
plt.figure(figsize=(8, 4))
sns.histplot(forces, bins=40, color="green", kde=True)
plt.title("Average Force Magnitude per Sample (first 1k)")
plt.xlabel("Force Magnitude (eV/Å)")
plt.ylabel("Count")
plt.show()

# ---------------------------------------------------------------
# 4️⃣ Cell Volume Distribution
# ---------------------------------------------------------------
volumes = []
for sample in ds[:1000]:
    cell = np.array(sample["cell"])
    if cell.shape == (3, 3):
        volumes.append(abs(np.linalg.det(cell)))

plt.figure(figsize=(8, 4))
sns.histplot(volumes, bins=40, color="purple", kde=True)
plt.title("Cell Volume Distribution (first 1k)")
plt.xlabel("Volume (Å³)")
plt.ylabel("Count")
plt.show()

# ---------------------------------------------------------------
# 5️⃣ Visualize a Few Atomic Structures
# ---------------------------------------------------------------
from ase import Atoms
from ase.visualize import view

def show_structure(idx):
    sample = ds[idx]
    pos = np.array(sample["pos"])
    numbers = np.array(sample["atomic_numbers"])
    atoms = Atoms(numbers=numbers, positions=pos)
    print(f"Sample ID: {sample['sid']}")
    print(f"Energy (init, relaxed): {sample['y_init']:.3f}, {sample['y_relaxed']:.3f}")
    print(f"Atoms: {len(numbers)}, Cell Volume: {np.linalg.det(sample['cell']):.2f} Å³")
    plot_atoms(atoms, rotation=('90x,0y,0z'), show_unit_cell=0)
    plt.show()

# View a few random structures
for idx in [0, 10, 50, 100]:
    show_structure(idx)

# ---------------------------------------------------------------
# 6️⃣ Summary Statistics
# ---------------------------------------------------------------
summary = {
    "Total samples": len(ds),
    "Mean initial energy": np.mean(y_init),
    "Mean relaxed energy": np.mean(y_relaxed),
    "Mean force magnitude": np.mean(forces),
    "Mean cell volume": np.mean(volumes),
}
print("📊 Dataset Summary:")
for k, v in summary.items():
    print(f"{k}: {v:.4f}" if isinstance(v, (float, int)) else f"{k}: {v}")
